<a href="https://colab.research.google.com/github/Tokoloshe714/26COA122CW/blob/toolRent/menu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
"""
Most of the following i got from AI as i did not know how to remove the sample data folder that appears automatically every new runtime. I need to remove the file as I am not permitted more than 10 files for submition.
Note this is not directly intended to be part of my coursework, but is needed."""


def remove_colab_sample_data() -> None:
    import shutil
    shutil.rmtree(paths["Unwanted"], ignore_errors=True)

In [17]:
"""
toolSearch and associated functions

Last edited on: 2026/11/14
            By: F613550
Cell description:
    This cell contains *four(4)* functions. see function marked ** for more info.

    • search_setup reads all the tools from the file Tool_Info.txt line by
      line. Each tool is stored as an element of a list that has the tool attributes in a dictionary.
    • find_query searches the tool list for matches against a chosen attribute
      using fuzzy string matching and returns the best results.
    • if_new_query is the callback that runs whenever the search box or
      dropdown changes. It clears the previous results and displays the new matches.
    • toolSearch builds the interactive search UI and connects the widgets so
      the search updates live.
"""

def populate_tool_info_list(file):
    for line in file:
        line = line.strip()
        if not line: # accounts for empty lines
            continue
        line_info = [att.strip() for att in line.split(",")]
        if len(line_info) != 5:
            return "Missing data for a tool"
        tool_info_list.append({
            "ID": line_info[0],
            "Name": line_info[1],
            "Size": line_info[2],
            "Purchase Price": line_info[3],
            "Purchase Date": line_info[4],
            })


def tool_info_setup() -> list:
    global tool_info_list
    # Define a list[dict{}]. Better than many r/w to .txt file
    tool_info_list = []
    try:
        with open(paths["Tools"], "r") as file:
            populate_tool_info_list(file)
    except error as e:
        print(f"There was an error: {e}")
        return e
    return tool_info_list


def if_new_query(change, tools_found) -> None:
    with tools_found:
        clear_output(wait=True)
        query = search_box.value
        attribute = attribute_dropdown.value
        results = toolSearch(query, attribute)
        if not results:
            print("No matches found.")
            return
        import pandas as pd
        display(pd.DataFrame(results))


def toolSearch(search_string: str,attribute: str, limit: int= 5) -> list:
    if not search_string.strip(): # i might change this as it may be annoying
        return tool_info_list[:limit]
    option = [str(tool[attribute]) for tool in tool_info_list]
    # search for all items in tool_info_list according to options
    sorted_matches = process.extract(
        search_string,
        option,
        scorer=fuzz.WRatio, # method for string comparison
        limit=limit
    )
    # reformatting
    results = []
    for match_test, score, index in sorted_matches:
        tool = tool_info_list[index].copy()
        #tool["Score"] = score
        results.append(tool)
    # return a sorted list of all valid tools in the correct TYPE!!!
    return results



In [18]:
"""
toolRent
Last edited on: 2026/11/14
             By: F613550
Cell description:

    • Customers should be identified using their unique ID numbers. For
      simplicity, we use 4-digit integers (e.g.,1000-9999) for these IDs. Only customers with active    subscriptions, as verified by the provided
    • ‘subscriptionManager.pyc’ module, are considered valid for renting tools
      Your program should distinguish between valid and invalid IDs
    • Prompts the store manager for the customer's ID and the ID of the tool(s)
      they wish to rent.
    • Rental_History.txt: Stores initial example data for the rental history of
      tools. Populate this text file with realistic data (minimum 50 records).
      You may write a program to generate this data.
    •To rent a tool, the store manager should provide a customer ID and the tool's ID number. Your program should then:
    1. Validate the input and use the provided ‘subscriptionManager.pyc’ to check the customer's subscription status. Return an error message if either is not valid.
    2. Depending on the tool's availability and the customer's subscription limit, either allow the manager to rent the tool by updating the related records in the database or return an appropriate message indicating why the rental cannot proceed
"""


def subscription_setup() -> None: # Load the subscription data
    global subscriptions
    subscriptions = smMAJ.load_subscriptions(paths["Subscription"])
    return


def valid_customer(customer_id: str) -> bool:
    global subscriptions
    # error checker
    if len(customer_id) == 4:
        premium = smMAJ.check_subscription(customer_id, subscriptions)
        return premium # bool
    return


def currently_rented() -> bool: # is used by rent and return for validation
    # basic idea
    # return true if the last entry of it has no return date str(None)
    # false otherwise
    return # will make at a later point


def valid_tool(tool_id: int) -> tuple[bool, bool]: # rent, return
    global tool_info_list
    RENTABLE = True
    RETURNABLE = True
    if tool_id not in tool_info_list.get("ID"):
        return not RENTABLE, not RETURNABLE   # tool does not exist
    if not currently_rented():
        return RENTABLE, not RETURNABLE       # valid for renting out
    else:
        return not RENTABLE, RETURNABLE       # valid for returning


def get_tool_name(tool_id) -> str: # used by rent
    global tool_info_list
    for tool in tool_info_list:
        if tool["ID"] == tool_id:
            return tool["Name"]
    return None


def rent_out_tool(customer_id, tool_id) -> bool:
    global rental_path
    # for now we will just rent out for 2 weeks from todays date
    try:
        with open(rental_path, "a") as file:
            tool_name = get_tool_name(tool_id)
            start_date = date.today()
            end_date = str(None)
            new_line = f"{tool_name},{start_date},{end_date},{customer_id}\n"
            file.write(new_line)
            return True
    except error as e:
        print(f"An error {e} occured.")
        return False


def toolRent(customer_id: str, tool_id: str) -> None:
    # Validation
    if not valid_customer(customer_id):
        print("Invalid customer ID or does not have an active subscription.")
        return
    if not valid_tool: #tool available?
        print("Invalid tool ID or tool is already rented out")
        return
    # rent tool to customer
    success = rent_out_tool(customer_id, tool_id)
    if not success:
        print("Failed to update rental history.")
        return
    print(f"Tool {tool_id} has been rented to customer {customer_id}.")


def on_rent_clicked(b, output) -> None:
    with output:
        clear_output()
        customer_id = customer_box.value.strip()
        tool_id = tool_box.value.strip()
        if not customer_id or not tool_id:
            print("Please enter both Customer ID and Tool ID.")
            return
        toolRent(customer_id, tool_id)



In [19]:
"""
toolReturn
Last edited on: 2026/11/14
             By: F613550
Cell description:
    •functions that prompt the store manager for the ID of the tool(s) they wish to return. If the ID is invalid, or the tool is not returnable, the program should return an error message, or a confirmation message.
    • The database (txt file) should be updated accordingly.
    •Feedback will be stored in an external database and managed by feedbackManager.pyc.’
"""

def tool_overdue() -> bool: # will make at a later point
    return False


def update_return_date(rental_path, tool_id) -> bool:
    # updates the return date to today
    return True


def feedback_setup() -> None:
    global feedback_list
    feedback = fmMAJ.load_feedback(paths["Feedback"])


def give_tool_feedback(tool_id):
    rating = str(input("Give our tool a rating from 1 to 10"))
    while not rating.isdigit(1, 10):
        rating = str(input("ERROR: Rating must be 1 to 10"))
    comments = str(input("Input any additional comments."))
    if comments == "":
        comments = "NO COMMENT ON RETURN!"
    fmMAJ.addfeedback(str(tool_id), rating, comments, paths["Feedback"])


def toolReturn(tool_id: str) -> None:
    RENTABLE, RETURNABLE = valid_tool()
    if not RETURNABLE:
        print("Cannot return as tool was not rented, or an error has occured")
        return
    days_overdue = tool_overdue()
    if tool_overdue():
        print(f"Tool is overdue by {days_overdue}")
    success = update_return_date(tool_id)
    if not success:
        print("Failed to process return.")
        return
    print(f"Tool {tool_id} returned.")
    give_tool_feedback(tool_id)


def on_return_clicked(b, output) -> None:
    with output:
        clear_output()
        customer_id = customer_box.value.strip()
        tool_id = tool_box.value.strip()
        if not tool_id:
            print("Please enter Tool ID.")
            return
        toolReturn(tool_id)



In [20]:
"""
menu

Last edited on: 2026/11/14
             By: F613550
Cell description:
    This cell contains the main program flow and supporting functions.
    • Installs required packages and imports all necessary libraries.
    • upload_files checks whether the required data files are already present.
    • setup runs the one-time initialisation steps
    • manager_login currently just returns True (placeholder for future login).
    • menu starts the program by calling the login check and then launching
      the tool search interface.
    • Calling setup() and menu() to run the program.
"""

# INSATLATIONS, IMPORTS AND GLOBAL CONSTANTS************************************

!pip install -q rapidfuzz ipywidgets

from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output
from rapidfuzz import process, fuzz
from os import error
from pathlib import Path
from datetime import date, timedelta

global paths, rental_period

paths = {
    "Unwanted": "/content/sample_data/",
    "SManager": "subscriptionManager.pyc",
    "Subscription": "/content/Subscription_Info.txt",
    "FManager": "feedbackManager.pyc",
    "Feedback": "/content/Tool_Feedback.txt",
    "Bookings": "/content/Bookings.txt",
    "Rental": "/content/Rental_History.txt",
    "Tools": "/content/Tool_Info.txt"
}

rental_period = timedelta(weeks= 2)

# SETUP FUNCTIONS***************************************************************

def upload_files():
    try:
        uploaded = files.upload()
        if not uploaded:
            print("Upload was cancelled or no files were selected.")
            return False
        for filename in uploaded.keys():
            print(f"File {filename} uploaded successfully.")
        return True
    except Exception as e:
        print(f"Error {e} occurred during the upload.")
        print("Please check the file name and location and try again.")


def file_setup() -> None: # Upload all files needed with error handling
    f_needed = [
       paths["SManager"],
       paths["Subscription"].replace("/content/", ""),
       paths["FManager"],
       paths["Feedback"].replace("/content/", ""),
       paths["Bookings"].replace("/content/", ""),
       paths["Tools"].replace("/content/", "")
    ]
    while True:
        # Check for missing files
        missing_f = [f for f in f_needed if not Path(f"/content/{f}").exists()]
        if not missing_f:
            print("All required files are already present. Skipping upload.")
            break
        print("Missing files:", missing_f)
        print("Please upload the missing files...")
        success = upload_files()
        if not success:
            break


def setup() -> None:    #is first thing to run and is only run once
    global smMAJ, fmMAJ
    remove_colab_sample_data()
    file_setup()
    tool_info_setup()
    # imports need to be global and files must be uploaded first
    import subscriptionManager as smMAJ
    import feedbackManager as fmMAJ
    subscription_setup()
    #feedback_setup()


# SECURITY**********************************************************************

def manager_login() -> bool:
    return True


# USER INTEFACE FUNCTIONS*******************************************************

def search_ui() -> None:
    global search_box, attribute_dropdown
    tool_attributes = ["ID", "Name", "Size", "Purchase Price",  "Purchase Date"]
    search_box = widgets.Text(
        value="",
        placeholder="Type to search...",
        description="Search:",
        continuous_update=True, # updates live while typing
        layout=widgets.Layout(width="400px")
    )
    attribute_dropdown = widgets.Dropdown(
        options=tool_attributes,
        value="Name",
        description="Search in:",
        layout=widgets.Layout(width="250px")
    )
    tools_found = widgets.Output()
    # dont like deep nesting. going back to lambda
    search_box.observe(lambda change: (change, tools_found), names="value")
    attribute_dropdown.observe(lambda change: (change, tools_found), names="value")
    search_interface = widgets.VBox([
        widgets.HTML("Tool Search"),
        widgets.HBox([search_box, attribute_dropdown]), tools_found
        ])
    display(search_interface)
    if_new_query(None, tools_found=tools_found)


def rental_ui() -> None: # may have to add lambda to pass variables. TEST FIRST!
    global customer_box, tool_box
    customer_box = widgets.Text(
        value="",
        placeholder="Enter Customer ID",
        description="Customer ID:",
        layout=widgets.Layout(width="350px")
    )
    tool_box = widgets.Text(
        value="",
        placeholder="Enter Tool ID",
        description="Tool ID:",
        layout=widgets.Layout(width="350px")
    )
    rent_button = widgets.Button(
        description="Rent Tool",
        button_style="success",
        layout=widgets.Layout(width="160px")
    )
    return_button = widgets.Button(
        description="Return Tool",
        button_style="warning",
        layout=widgets.Layout(width="160px")
    )
    output = widgets.Output()
    rent_button.on_click(on_rent_clicked())
    return_button.on_click(on_return_clicked())
    buttons = widgets.HBox([rent_button, return_button])
    ui = widgets.VBox([
        widgets.HTML("Tool Rental / Return"),
        customer_box,
        tool_box,
        buttons,
        output
    ])
    display(ui)


# MAIN MENU*********************************************************************

def menu() -> None:

    manager_login()
    search_ui()
    rental_ui()


# RUN SYSTEM********************************************************************

setup()
menu()

All required files are already present. Skipping upload.


TypeError: on_rent_clicked() missing 2 required positional arguments: 'b' and 'output'